# Tutorial 22: Deep Agents

In this tutorial we explore Deep Agents — the official LangChain abstraction announced in July 2025 for building agents that go beyond simple tool-calling loops. The `deepagents` package provides the architectural pattern behind products like Deep Research, Manus, and Claude Code.

## 1. What Makes an Agent "Deep"?

Harrison Chase (LangChain CEO) coined the term to describe agents that combine four pillars:

| Pillar | Description | Why it matters |
|---|---|---|
| **Detailed system prompt** | Long, opinionated prompt with few-shot examples | Sets agent behaviour precisely |
| **Planning tool** | `write_todos` — a no-op that lets the agent plan before acting | Prevents impulsive, unstructured execution |
| **Sub-agents** | Spawn isolated child agents via the `task` tool | Context isolation, parallel specialisation |
| **Filesystem** | `read_file`, `write_file`, `edit_file` | Persistent memory and inter-agent data sharing |

The `deepagents` package is built on top of LangGraph and returns a standard runnable that supports streaming, HITL, and memory.

## 2. Setup

In [1]:
# Install: pip install deepagents langchain-groq
import os
from deepagents import create_deep_agent
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage

# init_chat_model() is provider-agnostic — covered in detail in Tutorial 23
model = init_chat_model("groq:meta-llama/llama-4-scout-17b-16e-instruct", temperature=0.1)

print("Setup complete.")

[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


Setup complete.


## 3. Minimal Deep Agent

`create_deep_agent()` with no arguments creates a fully featured agent with all built-in tools and a default system prompt.

In [2]:
# Minimal setup — uses built-in tools and default prompt
agent = create_deep_agent(model=model)

print("Deep agent created.")
print("Built-in tools available to the agent:")
for t in agent.get_graph().nodes:
    print(f"  - {t}")

Deep agent created.
Built-in tools available to the agent:
  - __start__
  - model
  - tools
  - TodoListMiddleware.after_model
  - PatchToolCallsMiddleware.before_agent
  - __end__


## 4. Built-in Tools

Every Deep Agent comes with a set of built-in tools. You can add custom tools on top.

In [3]:
# The built-in tools and their purposes:
builtin_tools = {
    "write_todos": "Planning — the agent breaks its task into tracked steps before acting",
    "read_file": "Read a file from the virtual filesystem (text, PDF, images in v0.5+)",
    "write_file": "Write content to a file (agent's primary output mechanism)",
    "edit_file": "Edit a specific section of an existing file",
    "list_files (ls)": "List files in a directory",
    "grep": "Search file content with a pattern",
    "task": "Spawn a synchronous sub-agent with its own isolated context window",
}

for name, purpose in builtin_tools.items():
    print(f"  {name:25s}: {purpose}")

  write_todos              : Planning — the agent breaks its task into tracked steps before acting
  read_file                : Read a file from the virtual filesystem (text, PDF, images in v0.5+)
  write_file               : Write content to a file (agent's primary output mechanism)
  edit_file                : Edit a specific section of an existing file
  list_files (ls)          : List files in a directory
  grep                     : Search file content with a pattern
  task                     : Spawn a synchronous sub-agent with its own isolated context window


## 5. Custom Tools and System Prompt

In [4]:
@tool
def search_arxiv(query: str) -> str:
    """Search academic papers on ArXiv for a given query."""
    # In production this would call the ArXiv API
    return (
        f"ArXiv results for '{query}':\n"
        "1. 'Advances in Large Language Models' (2024) — Smith et al. — 847 citations\n"
        "2. 'Scaling Laws for Neural Language Models' (2023) — Jones et al. — 1,203 citations\n"
        "3. 'Chain-of-Thought Prompting Elicits Reasoning' (2022) — Wei et al. — 3,421 citations"
    )


@tool
def fetch_webpage(url: str) -> str:
    """Fetch the text content of a webpage."""
    return f"[Content of {url}]\nThis page contains detailed information about the requested topic. Key findings include significant research advances, practical applications, and ongoing challenges in the field."


research_agent = create_deep_agent(
    model=model,
    tools=[search_arxiv, fetch_webpage],
    system_prompt=(
        "You are a meticulous AI research assistant. When given a research topic:\n"
        "1. ALWAYS start by calling write_todos to plan your research steps\n"
        "2. Search for relevant academic papers and sources\n"
        "3. Synthesise findings into a structured report\n"
        "4. Save the final report to a file using write_file\n"
        "\nBe thorough, cite your sources, and organise your report with clear sections."
    )
)

print("Custom research agent created.")

Custom research agent created.


## 6. Running the Deep Agent

In [5]:
result = research_agent.invoke({
    "messages": [HumanMessage(content="What are 2 key capabilities of modern AI agents? Write a short summary.")]
})

print(result["messages"][-1].content[:500])

Modern AI agents have several key capabilities that enable them to perform complex tasks. Two of the most significant capabilities are:

1. **Autonomy**: Modern AI agents can operate independently, making decisions and taking actions without human intervention. They can learn from their environment, adapt to new situations, and adjust their behavior accordingly.
2. **Contextual Understanding**: AI agents can understand and interpret context, enabling them to respond appropriately to different si


## 7. Streaming the Agent's Progress

Since Deep Agents return a standard LangGraph runnable, you can stream all events in real time.

In [6]:
print("=== STREAMING AGENT EXECUTION ===")

for chunk in research_agent.stream(
    {"messages": [HumanMessage(content="Name the top 2 LLM papers. Be very brief.")]},
    stream_mode="updates"
):
    for node_name, node_output in chunk.items():
        if node_output is not None and node_name != "__end__":
            msgs = node_output.get("messages", [])
            for msg in msgs:
                content = str(getattr(msg, 'content', ''))[:100]
                if content:
                    print(f"[{node_name}] {content}")

=== STREAMING AGENT EXECUTION ===
[tools] ArXiv results for 'top LLM papers':
1. 'Advances in Large Language Models' (2024) — Smith et al. — 8
[model] The top 2 LLM papers are:

1. "Scaling Laws for Neural Language Models" (2023) by Jones et al.
2. "A


## 8. Sub-Agents — The `task` Tool

The `task` tool lets the main agent spawn isolated child agents for specialised subtasks. Each sub-agent has its own context window — it cannot see the parent's conversation history, which prevents context pollution.

In [7]:
# The agent uses the 'task' tool automatically when it decides to delegate
# Here we show conceptually how the parent orchestrates sub-agents:

print("""
How Deep Agent sub-agent delegation works:

Parent Agent:
  1. Calls write_todos: ['Search papers', 'Analyse results', 'Write report']
  2. Calls task(prompt='Search for papers on X', tools=[search_arxiv])
     → Sub-Agent A runs with isolated context, returns search results
  3. Calls task(prompt='Analyse these papers: {results}', tools=[])
     → Sub-Agent B runs with isolated context, returns analysis
  4. Calls write_file('report.md', combined_synthesis)
  5. Returns final answer to user

Benefits:
  - Sub-agents cannot accidentally see sensitive data from other steps
  - Each sub-agent's context window is fresh (no token waste)
  - Sub-agents can be specialised with different tools/prompts
  - In v0.5+: async tasks run sub-agents in parallel
""")


How Deep Agent sub-agent delegation works:

Parent Agent:
  1. Calls write_todos: ['Search papers', 'Analyse results', 'Write report']
  2. Calls task(prompt='Search for papers on X', tools=[search_arxiv])
     → Sub-Agent A runs with isolated context, returns search results
  3. Calls task(prompt='Analyse these papers: {results}', tools=[])
     → Sub-Agent B runs with isolated context, returns analysis
  4. Calls write_file('report.md', combined_synthesis)
  5. Returns final answer to user

Benefits:
  - Sub-agents cannot accidentally see sensitive data from other steps
  - Each sub-agent's context window is fresh (no token waste)
  - Sub-agents can be specialised with different tools/prompts
  - In v0.5+: async tasks run sub-agents in parallel



## 9. Deep Agent with HITL and Memory

Deep Agents support the full LangGraph feature set — including Human-in-the-Loop and long-term memory.

In [8]:
from langgraph.checkpoint.memory import MemorySaver

persistent_agent = create_deep_agent(
    model=model,
    tools=[search_arxiv],
    system_prompt="You are a research assistant. Always plan first with write_todos.",
    checkpointer=MemorySaver()
)

config = {"configurable": {"thread_id": "research-session-1"}}

# Turn 1
r1 = persistent_agent.invoke(
    {"messages": [HumanMessage(content="What is AI safety in one sentence?")]},
    config
)
print("Turn 1:", r1["messages"][-1].content[:200])

# Turn 2 — agent remembers the topic
r2 = persistent_agent.invoke(
    {"messages": [HumanMessage(content="Name 2 key papers on that topic.")]},
    config
)
print("\nTurn 2:", r2["messages"][-1].content[:200])

Turn 1: AI safety refers to the practices and techniques used to ensure that artificial intelligence systems operate in a way that is safe, reliable, and beneficial to humans, while minimizing the risk of har

Turn 2: Two key papers on AI safety are:

1. "Concrete Problems in AI Safety" by Amodei et al. (2016)
2. "Aligning Superintelligent AI with Human Values" by Yudkowsky (2015)

These papers are considered found


## 10. Conclusion

In this tutorial we explored Deep Agents:
- `create_deep_agent(model, tools, system_prompt)` creates a fully featured agent on top of LangGraph
- Built-in tools: `write_todos` (planning), file operations (memory), `task` (sub-agents)
- Provider-agnostic via `init_chat_model()` — swap models without changing code
- Returns a standard LangGraph runnable: supports `.invoke()`, `.stream()`, HITL, checkpointing
- Sub-agents have isolated context windows — preventing context pollution

In Tutorial 23 we cover the LangChain 1.0 utilities that power `init_chat_model()`, `trim_messages()`, and structured output.